# SignalScope Analysis Report

**Dataset:** synthetic_momentum_v1  
**Source:** momentum  
**Type:** synthetic  


## Data Description

This dataset was generated using a synthetic data generating process (DGP). The signal is embedded directly into returns using a linear factor model.

**Model:** Linear Factor Model  
**Equation:** `r_{i,t} = β · s_{i,t} + ε_{i,t}`  

**Implications:**
- Predictive power is structurally embedded
- IC and Sharpe may be inflated
- Results are not representative of real market data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline


In [ ]:
# Dataset (first rows from the processed pipeline)
_rows = [
    {
        "date": "2020-01-01",
        "asset": "ASSET_000",
        "signal": 0.08249430428370294,
        "return": 0.02010410713556869
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_001",
        "signal": 0.05051506297463688,
        "return": 0.022016827089203696
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_002",
        "signal": -1.7567905055789348,
        "return": -0.5101928356622852
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_003",
        "signal": -0.4578428392637714,
        "return": -0.1433170527251869
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_004",
        "signal": -1.046967562282426,
        "return": -0.3047723484567798
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_005",
        "signal": 0.6749804835796053,
        "return": 0.2149385570919026
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_006",
        "signal": 0.893087422223549,
        "return": 0.2705562760921035
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_007",
        "signal": 0.3285178485491658,
        "return": 0.10790779150549844
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_008",
        "signal": -0.8776129932016701,
        "return": -0.26374285884435006
    },
    {
        "date": "2020-01-01",
        "asset": "ASSET_009",
        "signal": 0.38187174018524606,
        "return": 0.11003613231201574
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_000",
        "signal": 0.7216648816031227,
        "return": 0.21297783040831705
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_001",
        "signal": 0.6727970245601584,
        "return": 0.20324534031035868
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_002",
        "signal": 0.4625606830102463,
        "return": 0.12359167575610358
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_003",
        "signal": -0.8602975472358647,
        "return": -0.24464357641652315
    },
    {
        "date": "2020-01-02",
        "asset": "ASSET_004",
        "signal": 0.1780863208373457,
        "return": 0.05261271339291425
    }
]

df = pd.DataFrame(_rows)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

df.head()


## Quantile Analysis


In [ ]:
# Quantile Analysis — mean return per signal bucket
_labels = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
_values = [-0.413659, -0.163496, -0.006293, 0.160191, 0.419099]

if _labels and _values:
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#d73027' if v < 0 else '#1a9850' for v in _values]
    ax.bar(_labels, _values, color=colors, edgecolor='white')
    ax.set_title('Quantile Returns (mean forward return per bucket)')
    ax.set_xlabel('Signal Quantile')
    ax.set_ylabel('Mean Return')
    plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
    # Spread (Q5 - Q1): 0.832758
    plt.tight_layout()
    plt.show()
else:
    print('No quantile data available.')


## Signal Structure


In [ ]:
# Signal vs Return scatter plot with OLS trend line
if 'signal' in df.columns and 'return' in df.columns:
    plt.figure(figsize=(5, 4))
    plt.scatter(df['signal'], df['return'], alpha=0.6, color='steelblue', s=18)

    # Regression trend line
    try:
        z = np.polyfit(df['signal'], df['return'], 1)
        p = np.poly1d(z)
        x_sorted = np.sort(df['signal'].values)
        plt.plot(x_sorted, p(x_sorted), color='tomato', linewidth=1.5, label='OLS fit')
        plt.legend()
    except Exception:
        pass

    plt.xlabel('Signal')
    plt.ylabel('Return')
    plt.title('Signal vs Return')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print('signal or return column not found in df.')


*The **linear pattern** reflects strong factor exposure (beta-driven returns). The OLS trend line aligns with a high-beta, low-alpha regime.*


## Factor Decomposition


In [ ]:
# Factor Decomposition — OLS regression results
# Equation: return = alpha + beta * signal + epsilon

beta         = 3.327823943902426
alpha        = 0.0002890948446926068
residual_std = 0.03313099846709294

print(f'Beta:         {beta:.4f}' if beta is not None else 'Beta:         N/A')
print(f'Alpha:        {alpha:.4f}' if alpha is not None else 'Alpha:        N/A')
print(f'Residual std: {residual_std:.4f}' if residual_std is not None else 'Residual std: N/A')

# Interpretation:
# High beta + near-zero alpha → signal is primarily factor-driven
# Positive alpha → signal carries independent predictive content


## Interpretation

The Information Coefficient (IC) and Rank IC values near 1 indicate exceptionally strong linear and monotonic predictive power of the signal for forward returns, well above the meaningful threshold of 0.1. The Quantile Spread (Q5 − Q1) of 0.8328 suggests a substantial economic separation between the top and bottom signal buckets, supporting strong cross-sectional ordering. The Cross-Sectional IC Stability is very high with a low standard deviation, indicating consistent temporal stability of the signal’s predictive ability. The Factor Beta of 3.3278 is large, while the Factor Alpha is near zero at 0.0003, implying that the signal’s returns are primarily driven by factor exposure rather than independent alpha generation. Overall, the signal reflects strong factor-driven performance with limited evidence of independent alpha.


## Signal Classification

**Type:** factor-driven  
**Confidence:** high  

This signal primarily captures **systematic factor exposure** rather than independent alpha.


## Conclusion

The signal appears to derive its performance from **factor exposure** rather than independent alpha. Returns are largely explained by systematic risk.  

**Classification:** factor-driven  
**Confidence:** high  


---

*This notebook was generated automatically by SignalScope.*
